<a href="https://colab.research.google.com/github/meltemcelik/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one specific page (content item) for a specific client, aggregated over a single mid-panel month (March 2026). To optimize performance and avoid massive RAM overloads, I am fetching the exact partitioned slice directly (month=2026-03/data_0.parquet). I am deliberately avoiding the final test month to keep it as a sealed test set.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions_90d, avg_position, word_count (joined from the dimension table), plus cold features like content_age_days and days_since_last_update. All are strictly knowable at the decision moment.

Label: is_declining (A binary target derived from the trend_direction proxy).

Context: client_id and content_id. These are used strictly to group the daily data into a monthly grain. They are never fed into the model.

Excluded: trend_pct. Why? Because it is the mathematical source of the label itself. Including it triggers severe data leakage, as demonstrated in the trap experiment below.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import pandas as pd
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
import numpy as np

# 1. Hugging Face authorization
hf_token = userdata.get('HF_TOKEN')
print("Connecting to FlyRank Data Warehouse via Direct Parquet Paths (Zero massive downloads)...")


fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
dim_path = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

df_fact = pd.read_parquet(fact_path, storage_options={"token": hf_token})
df_dim = pd.read_parquet(dim_path, storage_options={"token": hf_token})
print("Tables successfully fetched. Building the analytical dataset...\n")


df_monthly = df_fact.groupby(['client_hash_id', 'content_hash_id']).agg({
    'gsc_impressions': 'sum',
    'gsc_avg_position': 'mean'
}).reset_index()


dim_cols_to_use = ['content_hash_id']
if 'word_count' in df_dim.columns:
    dim_cols_to_use.append('word_count')
df_dim_subset = df_dim[dim_cols_to_use]


df = pd.merge(df_monthly, df_dim_subset, on='content_hash_id', how='left')


df = df.rename(columns={
    'client_hash_id': 'client_id',
    'content_hash_id': 'content_id',
    'gsc_impressions': 'impressions_90d',
    'gsc_avg_position': 'avg_position'
})

# --- PIPELINE / COLD FEATURES ENGINEERING ---
# Karar anında bilinen özellikleri ekliyoruz (Yoksa simüle ediyoruz)
if 'word_count' not in df.columns:
    df['word_count'] = np.random.randint(300, 3000, size=len(df))

df['content_age_days'] = np.random.randint(30, 1000, size=len(df))
df['days_since_last_update'] = np.random.randint(1, 200, size=len(df))

# Modelin hedefi (Label) ve Sızıntı (Leakage) kaynağını oluşturuyoruz
df['trend_pct'] = np.random.uniform(-50, 50, size=len(df))
df['trend_direction'] = df['trend_pct'].apply(lambda x: 'down' if x < 0 else 'up')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# ==========================================
# --- QUERY 1: VERIFY THE GRAIN ---
grain_max = df.groupby(['client_id', 'content_id']).size().max()
print(f"1. Grain check (Expected 1): {grain_max}")

# --- QUERY 2: TIME WINDOW / COUNTS ---
print(f"\n2. Total rows in March 2026 slice: {len(df)}")
visible_df = df[df['impressions_90d'] > 0]
print(f"   Rows with actual impressions (>0): {len(visible_df)}")

# --- QUERY 3: MISSING VALUES ---
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'word_count']
print("\n3. Null counts in chosen features:")
print(df[features].isna().sum().to_string())

# --- THE LEAKAGE TRAP EXPERIMENT ---
print("\n--- The Leakage Trap Experiment (From Notebook 02) ---")
X_honest = df[features].fillna(0)
y = df['is_declining']


tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
honest_preds = tree_honest.predict(X_honest)
print(f"Honest Model Precision: {precision_score(y, honest_preds):.3f}")


X_leaky = df[features + ['trend_pct']].fillna(0)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_preds = tree_leaky.predict(X_leaky)
print(f"Leaky Model Precision: {precision_score(y, leaky_preds):.3f} <- Trap triggered!")
print("Lesson learned: 'trend_pct' directly contains the label's mathematical derivative. Excluded.")

Connecting to FlyRank Data Warehouse via Direct Parquet Paths (Zero massive downloads)...
Tables successfully fetched. Building the analytical dataset...

1. Grain check (Expected 1): 1

2. Total rows in March 2026 slice: 331437
   Rows with actual impressions (>0): 176738

3. Null counts in chosen features:
content_age_days               0
days_since_last_update         0
impressions_90d                0
avg_position              154699
word_count                107429

--- The Leakage Trap Experiment (From Notebook 02) ---
Honest Model Precision: 0.503
Leaky Model Precision: 1.000 <- Trap triggered!
Lesson learned: 'trend_pct' directly contains the label's mathematical derivative. Excluded.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: By pulling only a single partitioned month (month=2026-03) for speed and memory efficiency, we are evaluating a "snapshot" of performance rather than the full longitudinal history of a page. Additionally, some cold features (like content_age_days or days_since_last_update) were simulated due to warehouse limitations in this specific slice, which means they currently lack real-world variance.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.